# Synthetic Dataset of Driver Telematics 

## Import packages

In [1]:
# Import libraries
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable
import dice_ml
from numpy.random import seed
seed(42)

# Fix seed
import random
random.seed(42)

In [2]:
import os
os.getcwd()

'/Users/agathefernandesmachado/Documents/PhD/counterfactuals_insurance/scripts_python'

## Import dataset

In [3]:
df_train = pd.read_csv("../out/df_train.csv")
df_calib = pd.read_csv("../out/df_calib.csv")
df_test = pd.read_csv("../out/df_test.csv")

In [4]:
df_train.head()

,Duration,Insured_age,Insured_sex,Car_age,Credit_score,Region,Pct_drive_rush,Accel_14miles,Brake_14miles,Y
0,0.997268,57,Female,3,849,Urban,mostly_pm,0,0,0
1,0.997268,30,Male,5,829,Urban,both,0,0,0
2,0.997268,30,Female,7,727,Urban,mostly_pm,0,0,0
3,0.997268,28,Male,9,715,Urban,both,1,1,0
4,0.997268,32,Female,4,796,Urban,both,0,0,0


In [5]:
df_train.shape

(58834, 10)

## Frequency model: GLM Poisson

In [6]:
# Import fitted model from R
glm_design_matrix_train = pd.read_csv("../../counterfactuals_insurance/out/glm_design_matrix_train.csv")
glm_coef = pd.read_csv("../../counterfactuals_insurance/out/glm_coefficients.csv")

In [7]:
glm_coef

,name,value
0,(Intercept),-0.442232
1,Insured_age,-0.005908
2,Insured_sexMale,-0.035216
3,Car_age,-0.090462
4,Credit_score,-0.002771
5,RegionUrban,0.216568
6,Pct_drive_rushmostly_am,-0.131413
7,Pct_drive_rushmostly_pm,0.324192
8,Pct_drive_rushboth,0.115399
9,Accel_14miles1,0.206094


In [8]:
def preprocess_glm(df):
    glm_df = pd.get_dummies(df_test, dtype=int)
    glm_df = glm_df.rename(
        columns={
            "Accel_14miles": "Accel_14miles1", 
            "Brake_14miles": "Brake_14miles1",
            "Insured_sex_Male": "Insured_sexMale",
            "Region_Urban": "RegionUrban",
            "Pct_drive_rush_both": "Pct_drive_rushboth",
            "Pct_drive_rush_mostly_am": "Pct_drive_rushmostly_am",
            "Pct_drive_rush_mostly_pm": "Pct_drive_rushmostly_pm"
            }
        )
    glm_df = glm_df[
        ['Insured_age', 'Insured_sexMale', 'Car_age', 'Credit_score', 'RegionUrban', 'Pct_drive_rushmostly_am',
        'Pct_drive_rushmostly_pm', 'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1', 'Y']
        ]
    return(glm_df)

In [9]:
def predict_glm_poisson(X, coef):
    df = X.copy()
    df["(Intercept)"] = 1
    beta = (
        coef
        .set_index("name")
        .reindex(df.columns)["value"]
        .values
        )  
    # Linear predictor
    eta = df.values @ beta #+ np.log(exposure)
    # Apply sigmoid
    rate = np.exp(eta)
    
    return np.array(rate) # Poisson mean

In [10]:
glm_df_test = preprocess_glm(df_test).drop(columns=['Y'])
glm_pred_test = predict_glm_poisson(glm_df_test, glm_coef)
glm_pred_test[0:5] # same as in R

array([0.05846351, 0.03961507, 0.03389885, 0.04190456, 0.01907873])

In [11]:
# Query instance
idx_query = 2357
print(glm_pred_test[idx_query])
query_instance_glm = glm_df_test.iloc[[idx_query]]

0.22406411659600484


In [12]:
query_instance_glm

,Insured_age,Insured_sexMale,Car_age,Credit_score,RegionUrban,Pct_drive_rushmostly_am,Pct_drive_rushmostly_pm,Pct_drive_rushboth,Accel_14miles1,Brake_14miles1
2357,48,1,3,602,1,0,1,0,1,1


In [13]:
class GLMWrapper:
    def __init__(self, coef):
        self.coef = coef

    def predict(self, X):
        return predict_glm_poisson(X, self.coef)

In [14]:
wrapped_model = GLMWrapper(glm_coef)
pred = wrapped_model.predict(glm_df_test)

In [15]:
wrapped_model.predict(query_instance_glm)

array([0.22406412])

In [16]:
# Categorical features
glm_df_train = preprocess_glm(df_train)
#categorical_features = glm_df_train.select_dtypes(include=['object', 'category']).columns.tolist()
#print(categorical_features)

In [17]:
# Continuous features except the outcome variable
continuous_features = glm_df_test.columns.tolist()
print(continuous_features)

['Insured_age', 'Insured_sexMale', 'Car_age', 'Credit_score', 'RegionUrban', 'Pct_drive_rushmostly_am', 'Pct_drive_rushmostly_pm', 'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1']


In [18]:
glm_df_train.columns

Index(['Insured_age', 'Insured_sexMale', 'Car_age', 'Credit_score',
       'RegionUrban', 'Pct_drive_rushmostly_am', 'Pct_drive_rushmostly_pm',
       'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1', 'Y'],
      dtype='str')

## DiCE

Documentation of the Python package: https://interpret.ml/DiCE/notebooks/DiCE_getting_started.html

In [18]:
# Define the DiCE data object
d = dice_ml.Data(dataframe = glm_df_train, continuous_features = continuous_features, outcome_name = 'Y')

In [19]:
# Create DiCE model object
m = dice_ml.Model(model=wrapped_model, backend='sklearn', model_type='regressor')  # use 'sklearn' trick for wrapper

In [20]:
telematics_variables = [
       'Pct_drive_rushmostly_am', 'Pct_drive_rushmostly_pm', 'Pct_drive_rushboth', 'Accel_14miles1', 'Brake_14miles1'
       ]

In [21]:
# Create explainer
genetic_explainer = dice_ml.Dice(d, m, method="genetic")
# Genetic method naturally handles categorical variables

In [ ]:
# Generate 3 counterfactuals aiming for smaller counts
# The only actionable factors are the telematics variables here
cf = genetic_explainer.generate_counterfactuals(
    query_instance_glm,
    total_CFs = 100, # max.
    desired_range=[0, 0.2],  # desired range
    features_to_vary = telematics_variables,
    initialization="kdtree",
    proximity_weight=0.2,
    diversity_weight=5.0,
    yloss_type="hinge_loss", 
    diversity_loss_type="dpp_style:inverse_dist",
    maxiterations=10, 
    thresh=1e-2
    )

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.24it/s]


In [23]:
# Show counterfactuals
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0.22406411170959473)


,Insured_age,Insured_sexMale,Car_age,Credit_score,RegionUrban,Pct_drive_rushmostly_am,Pct_drive_rushmostly_pm,Pct_drive_rushboth,Accel_14miles1,Brake_14miles1,Y
0,48,1,3,602,1,0,1,0,1,1,0.224064



Diverse Counterfactual set (new outcome: [0, 0.2])


,Insured_age,Insured_sexMale,Car_age,Credit_score,RegionUrban,Pct_drive_rushmostly_am,Pct_drive_rushmostly_pm,Pct_drive_rushboth,Accel_14miles1,Brake_14miles1,Y
0,-,-,-,-,-,-,0.0,-,-,-,0.1620232554608658
0,-,-,-,-,-,1.0,-,-,-,-,0.1964719268510517
0,-,-,-,-,-,-,-,-,0.0,-,0.18233362479300488
0,-,-,-,-,-,-,-,-,-,0.0,0.14176333262470334
0,-,-,-,-,-,-,-,-,0.0,0.0,0.11536082927014965
...,...,...,...,...,...,...,...,...,...,...,...
0,-,-,-,-,-,-,0.0,-,0.0,0.0,0.0834186990535193
0,-,-,-,-,-,-,0.0,-,-,-,0.1620232554608658
0,-,-,-,-,-,-,-,-,0.0,-,0.18233362479300488
0,-,-,-,-,-,-,0.0,1.0,-,-,0.18184212941277172


In [24]:
df_cf = cf.cf_examples_list[0].final_cfs_df

In [25]:
df_cf.head()

,Insured_age,Insured_sexMale,Car_age,Credit_score,RegionUrban,Pct_drive_rushmostly_am,Pct_drive_rushmostly_pm,Pct_drive_rushboth,Accel_14miles1,Brake_14miles1,Y
0,48,1,3,602,1,0,0,0,1,1,0.162023
0,48,1,3,602,1,1,1,0,1,1,0.196472
0,48,1,3,602,1,0,1,0,0,1,0.182334
0,48,1,3,602,1,0,1,0,1,0,0.141763
0,48,1,3,602,1,0,1,0,0,0,0.115361


In [26]:
df_cf.shape[0]

91

In [27]:
df_cf = df_cf.drop_duplicates().reset_index(drop=True)
df_cf.shape[0]

28

In [28]:
cols = [
    "Pct_drive_rushmostly_am",
    "Pct_drive_rushmostly_pm",
    "Pct_drive_rushboth"
]
# We need to ensure that the sum of the one-hot encoded telematics variables equals 1
df_cf = df_cf[df_cf[cols].sum(axis=1).isin([0, 1])]

In [29]:
df_cf = df_cf.rename(columns={"Y": "pred"})

In [30]:
df_cf.shape[0]

15

In [31]:
df_cf['idx_query'] = 2357+1

In [32]:
df_cf.to_csv("../out/cf_dice.csv", index=False)